# Statistics: rule-imposed vs. unconstrained speech

**Design:** 5 participants × 5 prompts (banana, table, tiger, tree, water) × 2 conditions (`rules`, `norules`), each recording about 30 s.
The unit of analysis is a **pair**: the same participant and prompt under both conditions. All effects are differences **rules − norules**.

**Sections**
1. Setup and data loading
2. Speech metrics (fluency, pausing, lexical)
3. Morphological metrics (synthesis, agglutination, fusion)
4. Paired tests: Wilcoxon signed-rank, d_z, Holm correction, and a mixed model with a participant random intercept
5. Figures: paired overview, forest plot, difference heatmaps

Input: `transcripts/labelled/*.txt` (reviewed verbatim transcripts) and `transcripts/verbatim_transcripts.csv` (durations).

## 1. Setup and data loading

In [ ]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

ROOT = Path("transcripts")
DATA = ROOT / "data"                   # MorphoLex + Morfessor cache
CONDS = ["norules", "rules"]
COLORS = {"norules": "tab:blue", "rules": "tab:orange"}
SEED = 0
pd.set_option("display.max_colwidth", 100)

In [ ]:
meta = pd.read_csv(ROOT / "verbatim_transcripts.csv")
rows = []
for f in sorted((ROOT / "labelled").glob("*.txt")):
    participant, session, prompt, condition = f.stem.split("_")
    rows.append(dict(id=f.stem, participant=participant, session=session, prompt=prompt,
                     condition=condition, raw=f.read_text().strip()))
df = pd.DataFrame(rows).merge(meta[["id", "duration_s"]], on="id", how="left")
df["no_speech"] = df.raw.str.contains(r"\[NO SPEECH", regex=True)

print(df.groupby(["participant", "condition"]).size().unstack(), "\n")
print("Excluded (no speech):", df.loc[df.no_speech, "id"].tolist())
df = df[~df.no_speech].reset_index(drop=True)

## 2. Speech metrics

| Metric | Definition |
|---|---|
| `n_words` | complete words (tags and partial words removed) |
| `speech_rate_wpm` | 60 × `n_words` / duration |
| `n_pauses`, `total_pause_s`, `mean_pause_s` | count, sum, mean of `[PAUSE x.xs]` (mean = 0 if no pauses) |
| `pause_frac` | `total_pause_s` / duration |
| `n_hes`, `hes_per_100w` | `[HESITATION: uh/um]` count, and per 100 words |
| `n_partial` | broken-off words (ending in `-`) |
| `type_token_ratio` | unique / total words |
| `mean_word_len` | letters per word |
| `mean_sent_len` | words per sentence (split on `.!?`) |

In [ ]:
TAG = re.compile(r"\[[^\]]*\]")
PAUSE = re.compile(r"\[PAUSE ([\d.]+)s\]")
HES = re.compile(r"\[HESITATION: \w+\]")
WORD = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*-?")

def clean_text(raw):
    return re.sub(r"\s+", " ", TAG.sub(" ", raw)).strip()

def speech_features(raw, dur):
    text = clean_text(raw)
    words = WORD.findall(text)
    full = [w.lower() for w in words if not w.endswith("-")]
    n = max(len(full), 1)
    pauses = [float(p) for p in PAUSE.findall(raw)]
    n_hes = len(HES.findall(raw))
    sents = [s for s in re.split(r"[.!?]+", text) if WORD.search(s)]
    return dict(
        text=text, n_words=len(full), speech_rate_wpm=60 * len(full) / dur,
        n_pauses=len(pauses), total_pause_s=sum(pauses),
        mean_pause_s=np.mean(pauses) if pauses else 0.0, pause_frac=sum(pauses) / dur,
        n_hes=n_hes, hes_per_100w=100 * n_hes / n,
        n_partial=sum(w.endswith("-") for w in words),
        type_token_ratio=len(set(full)) / n,
        mean_word_len=np.mean([len(w) for w in full]),
        mean_sent_len=len(full) / max(len(sents), 1),
    )

df = df.join(df.apply(lambda r: pd.Series(speech_features(r.raw, r.duration_s)), axis=1))
SPEECH_METRICS = ["n_words", "speech_rate_wpm", "n_pauses", "total_pause_s", "mean_pause_s", "pause_frac",
                  "n_hes", "hes_per_100w", "n_partial", "type_token_ratio", "mean_word_len", "mean_sent_len"]
df[["id"] + SPEECH_METRICS].head()

## 3. Morphological metrics

Each word is split into morphemes, and each boundary (juncture) is labelled:
- **agg**: clean concatenation (`help+ful`, `tree+s`)
- **ortho**: spelling-only change (`happy+ness → happiness`, `make+ing → making`)
- **fus**: fused or suppletive form (`went`, `are`, `decide+ion → decision`, `teeth`)

**Sources:** spaCy (`en_core_web_sm`) provides lemmas and inflectional features. MorphoLex-EN provides derivational segmentation. For words in neither, Morfessor (trained on MorphoLex words) is the fallback; its junctures are marked *unverified* and left out of the juncture indices.

| Index | Definition |
|---|---|
| `synthesis` | morphemes / word |
| `agglutination` | (agg + ortho) / verified junctures |
| `ortho_share` | ortho / verified junctures |
| `fusion_rate` | share of words with ≥ 1 fusional juncture |
| `multimorph_rate` | share of words with > 1 morpheme |

The `_c` suffix means content words only (NOUN, VERB, ADJ, ADV).

In [ ]:
import morfessor, spacy
nlp = spacy.load("en_core_web_sm")

SEGM_CSV = DATA / "morpholex_segm.csv"
if not SEGM_CSV.exists():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sheets = pd.read_excel(DATA / "MorphoLEX_en.xlsx", sheet_name=None)
    pd.concat([v[["Word", "POS", "Nmorph", "MorphoLexSegm"]] for v in sheets.values()
               if "MorphoLexSegm" in v.columns]).to_csv(SEGM_CSV, index=False)
mlex = pd.read_csv(SEGM_CSV).dropna(subset=["Word", "MorphoLexSegm"])
LEX = dict(zip(mlex.Word.astype(str).str.lower(), mlex.MorphoLexSegm))

MORF_BIN = DATA / "morfessor_model.bin"
mio = morfessor.MorfessorIO()
if MORF_BIN.exists():
    morf = mio.read_binary_model_file(str(MORF_BIN))
else:
    morf = morfessor.BaselineModel()
    morf.load_data([(1, w) for w in LEX if w.isalpha()])
    morf.train_batch(max_epochs=4)
    mio.write_binary_model_file(str(MORF_BIN), morf)

In [ ]:
SEG_RE = re.compile(r"<([^<>]+)<|\(([^()]+)\)|>([^<>]+)>")
INFL_POS = {"VERB", "AUX", "NOUN", "ADJ", "ADV", "PRON"}
CONTENT_POS = {"NOUN", "VERB", "ADJ", "ADV"}

def parse_morpholex(seg):
    return [next(g for g in m if g) for m in SEG_RE.findall(seg.replace("{", "").replace("}", ""))]

def ortho_variants(m):
    v = []
    if m.endswith("e"): v.append(m[:-1])
    if m.endswith("y"): v.append(m[:-1] + "i")
    if len(m) > 1 and m[-1] not in "aeiouwy": v.append(m + m[-1])
    return v

def classify_juncture(left, right, form):
    r = right[:3]
    if left + r in form: return "agg"
    if any(v + r in form for v in ortho_variants(left)): return "ortho"
    return "fus"

def classify_inflection(word, lemma, tok):
    if tok.pos_ not in INFL_POS: return None
    if word == lemma:
        past = "Past" in tok.morph.get("Tense") and tok.pos_ in {"VERB", "AUX"}
        plur = "Plur" in tok.morph.get("Number") and tok.pos_ == "NOUN"
        return "fus" if (past or plur) else None
    if word.startswith(lemma): return "agg"
    if any(word.startswith(v) for v in ortho_variants(lemma)): return "ortho"
    return "fus"

def analyse_token(tok):
    word, lemma = tok.text.lower(), tok.lemma_.lower()
    if tok.pos_ == "PROPN":
        morphs, src = [word], "propn"
    elif lemma in LEX or word in LEX:
        morphs, src = parse_morpholex(LEX[lemma if lemma in LEX else word]), "morpholex"
    else:
        morphs, src = morf.viterbi_segment(lemma)[0], "morfessor"
    base = word if src == "propn" else lemma
    junc = ([classify_juncture(a, b, base) for a, b in zip(morphs, morphs[1:])] if src == "morpholex"
            else ["unverified"] * (len(morphs) - 1))
    infl = classify_inflection(word, lemma, tok) if src != "propn" else None
    if infl: junc.append(infl)
    return dict(word=word, lemma=lemma, pos=tok.pos_, source=src,
                morphemes="+".join(morphs) + ("+INFL" if infl else ""),
                n_morphemes=len(morphs) + bool(infl), n_agg=junc.count("agg"), n_ortho=junc.count("ortho"),
                n_fus=junc.count("fus"), content=tok.pos_ in CONTENT_POS)

PARTIAL = re.compile(r"\b[A-Za-z']+-(?=\s|$)")
tokens = pd.DataFrame([dict(id=rid, **analyse_token(t))
                       for rid, text in zip(df.id, df.text)
                       for t in nlp(PARTIAL.sub(" ", text)) if t.is_alpha])
print(tokens.source.value_counts())

In [ ]:
def morph_indices(g):
    ver = g.n_agg.sum() + g.n_ortho.sum() + g.n_fus.sum()
    return pd.Series(dict(
        synthesis=g.n_morphemes.mean(),
        agglutination=(g.n_agg.sum() + g.n_ortho.sum()) / ver if ver else np.nan,
        ortho_share=g.n_ortho.sum() / ver if ver else np.nan,
        fusion_rate=(g.n_fus > 0).mean(),
        multimorph_rate=(g.n_morphemes > 1).mean()))

mi = tokens.groupby("id").apply(morph_indices)
mi = mi.join(tokens[tokens.content].groupby("id").apply(morph_indices).add_suffix("_c"))
MORPH_METRICS = list(mi.columns)
df = df.merge(mi, left_on="id", right_index=True)

print("Most frequent fusional words:")
tokens[tokens.n_fus > 0].groupby(["word", "morphemes"]).size().sort_values(ascending=False).head(10)

## 4. Paired tests

For each metric, the per-pair difference is `d = rules − norules`.
- **Wilcoxon signed-rank:** a non-parametric test of whether the median of `d` is 0. Holm correction is applied within each family (speech, morphology).
- **d_z:** mean(d) / sd(d), with a 95% bootstrap CI from 5,000 resamples of pairs.
- **Mixed model:** `d ~ 1 + (1 | participant)`. It tests the mean difference while allowing for repeated measures within a participant. With 4 participants the random-effect variance is poorly estimated, so treat this as a robustness check.

In [ ]:
ALL_METRICS = SPEECH_METRICS + MORPH_METRICS
wide = df.pivot_table(index=["participant", "prompt"], columns="condition", values=ALL_METRICS).dropna()
diffs = pd.DataFrame({m: wide[(m, "rules")] - wide[(m, "norules")] for m in ALL_METRICS})
print(f"{len(diffs)} complete pairs from {diffs.index.get_level_values('participant').nunique()} participants")

def bootstrap_dz(d, B=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (B, len(d)))]
    sd = bs.std(1, ddof=1)
    return np.nanpercentile(bs.mean(1) / np.where(sd > 0, sd, np.nan), [2.5, 97.5])

def mixed_p(d_series):
    dd = d_series.reset_index(name="d")
    if dd.d.std() == 0: return np.nan, np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = smf.mixedlm("d ~ 1", dd, groups=dd.participant).fit(reml=True)
    return fit.params["Intercept"], fit.pvalues["Intercept"]

rows = []
for m in ALL_METRICS:
    d = diffs[m].to_numpy()
    sd = d.std(ddof=1)
    lo, hi = bootstrap_dz(d) if sd > 0 else (np.nan, np.nan)
    est, p_mixed = mixed_p(diffs[m])
    rows.append(dict(metric=m, family="morphology" if m in MORPH_METRICS else "speech",
                     norules=wide[(m, "norules")].mean(), rules=wide[(m, "rules")].mean(),
                     mean_diff=d.mean(), dz=d.mean() / sd if sd > 0 else np.nan, dz_lo=lo, dz_hi=hi,
                     p_wilcoxon=stats.wilcoxon(d).pvalue if (d != 0).any() else np.nan,
                     p_mixed=p_mixed))
results = pd.DataFrame(rows).set_index("metric")
for fam, idx in results.groupby("family").groups.items():
    p = results.loc[idx, "p_wilcoxon"]
    ok = p.notna()
    results.loc[p[ok].index, "p_holm"] = multipletests(p[ok], method="holm")[1]
results.round(4)

In [ ]:
# Significant (uncorrected p < .05) and surviving Holm correction
results.loc[results.p_wilcoxon < 0.05, ["family", "norules", "rules", "dz", "dz_lo", "dz_hi",
                                         "p_wilcoxon", "p_holm", "p_mixed"]].round(4)

## 5. Figures

### 5a. Paired overview
The points are jittered sideways, and grey lines join the two recordings of each pair. Each side strip is a Gaussian fit: saturation shows density, the solid line is μ and the dashed lines are μ ± σ.

In [ ]:
def gaussian_strip(ax, values, x0, x1, color, ylim):
    mu, sd = np.mean(values), np.std(values, ddof=1)
    if not sd > 0: return
    y = np.linspace(*ylim, 300)
    dens = stats.norm.pdf(y, mu, sd)
    img = np.zeros((len(y), 1, 4)); img[..., :3] = to_rgb(color); img[:, 0, 3] = 0.85 * dens / dens.max()
    ax.imshow(img, extent=(x0, x1, *ylim), origin="lower", aspect="auto", zorder=0)
    ax.hlines(mu, x0, x1, color=color, lw=2, zorder=1)
    ax.hlines([mu - sd, mu + sd], x0, x1, color=color, lw=1.2, ls="--", zorder=1)

def paired_overview(metrics, title, ncols=4):
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.7 * ncols, 3.4 * nrows), squeeze=False)
    rng = np.random.default_rng(SEED)
    for ax, m in zip(axes.flat, metrics):
        a, b = wide[(m, "norules")].to_numpy(), wide[(m, "rules")].to_numpy()
        allv = np.concatenate([a, b]); pad = 0.15 * (np.ptp(allv) or 1)
        ylim = (allv.min() - pad, allv.max() + pad)
        gaussian_strip(ax, a, -0.55, -0.15, COLORS["norules"], ylim)
        gaussian_strip(ax, b, 1.15, 1.55, COLORS["rules"], ylim)
        xa, xb = rng.uniform(-0.12, 0.12, len(a)), 1 + rng.uniform(-0.12, 0.12, len(b))
        ax.plot(np.vstack([xa, xb]), np.vstack([a, b]), color="grey", alpha=0.3, lw=1, zorder=2)
        ax.scatter(xa, a, color=COLORS["norules"], s=16, alpha=0.8, zorder=3)
        ax.scatter(xb, b, color=COLORS["rules"], s=16, alpha=0.8, zorder=3)
        ax.set_xlim(-0.6, 1.6); ax.set_ylim(ylim); ax.set_xticks([0, 1], CONDS); ax.set_title(m, fontsize=10)
    for ax in axes.flat[len(metrics):]: ax.axis("off")
    fig.suptitle(title); fig.tight_layout()

paired_overview(SPEECH_METRICS, "Speech metrics", ncols=6)

In [ ]:
paired_overview(MORPH_METRICS, "Morphological indices", ncols=5)

### 5b. Forest plot
The black dot and bar show d_z and its 95% bootstrap CI; they are black when the CI excludes 0 and grey otherwise. A ★ marks metrics that survive Holm correction. The coloured dots show each participant's mean difference on the same scale.

In [ ]:
forest = results.dropna(subset=["dz"]).iloc[::-1]
part_means = diffs.groupby(level="participant").mean()
PCOL = dict(zip(part_means.index, plt.cm.Dark2.colors))
rng = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(8.5, 0.38 * len(forest) + 1.2))
for y, (m, r) in enumerate(forest.iterrows()):
    sig = r.dz_lo > 0 or r.dz_hi < 0
    col = "black" if sig else "grey"
    ax.hlines(y, r.dz_lo, r.dz_hi, color=col, lw=2.2 if sig else 1.4)
    ax.plot(r.dz, y, "o", color=col, ms=7, zorder=3)
    if r.p_holm < 0.05: ax.text(r.dz_hi + 0.05, y, "★", va="center")
    sd = diffs[m].std(ddof=1)
    for pname, v in part_means[m].items():
        ax.plot(v / sd, y + rng.uniform(-0.15, 0.15), "o", color=PCOL[pname], ms=4, alpha=0.75, zorder=2)
ax.axvline(0, color="black", lw=0.8)
ax.axhline((forest.family == "speech").to_numpy().argmax() - 0.5, color="lightgrey", ls=":")
ax.set_yticks(range(len(forest)), forest.index)
ax.set_xlabel("d_z   (← lower with rules  |  higher with rules →)")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=n) for n, c in PCOL.items()],
          title="participant mean", loc="lower right", fontsize=8)
ax.set_title("Effect of rules across all metrics"); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

### 5c. Difference heatmaps
Rows are participants and columns are prompts. Colour shows rules − norules (red = higher with rules), on a scale symmetric around 0 for each metric. The bold margins hold the per-prompt and per-participant means.

In [ ]:
def diff_heatmaps(metrics, title, ncols=4):
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.3 * nrows), squeeze=False)
    for ax, m in zip(axes.flat, metrics):
        mat = diffs[m].unstack("prompt")
        mat["mean"] = mat.mean(axis=1); mat.loc["mean"] = mat.mean(axis=0)
        v = np.nanmax(np.abs(mat.to_numpy())) or 1
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
        last_r, last_c = mat.shape[0] - 1, mat.shape[1] - 1
        for (i, j), val in np.ndenumerate(mat.to_numpy()):
            ax.text(j, i, f"{0 if abs(val) < 1e-9 else val:.2g}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(val) > 0.6 * v else "black",
                    fontweight="bold" if i == last_r or j == last_c else "normal")
        ax.axhline(last_r - 0.5, color="black", lw=1.2); ax.axvline(last_c - 0.5, color="black", lw=1.2)
        ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(mat.shape[0]), mat.index, fontsize=8)
        ax.set_title(m, fontsize=10); fig.colorbar(im, ax=ax, shrink=0.8)
    for ax in axes.flat[len(metrics):]: ax.axis("off")
    fig.suptitle(f"{title}  (rules − norules)"); fig.tight_layout()

diff_heatmaps(SPEECH_METRICS, "Speech metrics")

In [ ]:
diff_heatmaps(MORPH_METRICS, "Morphological indices", ncols=5)

## Notes and limitations
- **Small sample:** 20 pairs from 4 participants; Ranesh's rule recordings contain no speech. Treat uncorrected p-values as exploratory.
- **Duration:** taken from the CSV and includes any lead-in silence. The pipeline's `summary.csv` (MFA timings, speaking span only) would give more precise rates.
- **Morphology:** the juncture check is local. MorphoLex roots are given in underlying form, so Latinate allomorphy counts as fusional. spaCy tags pronoun case (`him → he`) as inflection. Clitics (`n't`, `'s`) are skipped.